In [1]:
import spacy
from spacy.matcher import PhraseMatcher # Sınıfı import ediyoruz
import pandas as pd
import time

# 1. Kütüphaneyi Doğru Import Et
try:
    from skillNer.skill_extractor_class import SkillExtractor
    from skillNer.general_params import SKILL_DB
    print("✅ 'skillNer' kütüphanesi başarıyla import edildi.")
except ImportError as e:
    print("❌ HATA: Kütüphane import edilemedi.")
    raise e

print("İP 1 - Adım 4 Başladı: Modeller yükleniyor...")

# 2. Spacy Modelini Yükle
try:
    nlp = spacy.load("en_core_web_lg")
except IOError:
    print("HATA: 'en_core_web_lg' modeli bulunamadı.")
    print("!python -m spacy download en_core_web_lg")
    raise

# 3. SkillExtractor'ı Başlat (KESİN ÇÖZÜM)
try:
    # DÜZELTME BURADA: 
    # 'PhraseMatcher(nlp.vocab)' yerine sadece 'PhraseMatcher' (Sınıfın kendisi) veriyoruz.
    skill_extractor = SkillExtractor(nlp, SKILL_DB, PhraseMatcher)
    print("✅ SkillExtractor başarıyla başlatıldı.")

except Exception as e:
    print(f"SkillExtractor başlatılırken hata: {e}")
    raise

# -------------------------------------------------------------------------
# ADIM 4.2: ADAY CV'SİNDEN BECERİLERİ ÇIKAR
# -------------------------------------------------------------------------
print("\nAdım 4.2: Aday CV metni işleniyor...")

try:
    with open("IP1_Ciktisi_Aday_Tam_Metin.txt", "r", encoding="utf-8") as f:
        cv_full_text = f.read()
except FileNotFoundError:
    print("HATA: Dosya bulunamadı. Önceki adımları kontrol edin.")
    raise

try:
    # SkillNER ile becerileri çıkar
    annotations = skill_extractor.annotate(cv_full_text)
    
    cv_skills_list = []
    
    # Tam eşleşmeler
    for match in annotations['results']['full_matches']:
        cv_skills_list.append(match['doc_node_value'])
        
    # Ngram eşleşmeler
    for match in annotations['results']['ngram_scored']:
        cv_skills_list.append(match['doc_node_value'])
        
    # Benzersiz yap ve temizle
    cv_skills_list = list(set([s.lower() for s in cv_skills_list]))
    
    print(f"-> CV'den {len(cv_skills_list)} adet beceri çıkarıldı.")
    print(f"Örnekler: {cv_skills_list[:5]}")
    
    # Kaydet
    pd.DataFrame(cv_skills_list, columns=["cikarilan_beceri_adi"]).to_pickle("IP1_Ciktisi_Aday_Cikarilan_Beceriler.pkl")

except Exception as e:
    print(f"CV işlenirken hata oluştu: {e}")

# -------------------------------------------------------------------------
# ADIM 4.3: PROJE İLANLARINDAN BECERİLERİ ÇIKAR
# -------------------------------------------------------------------------
print("\nAdım 4.3: Proje ilanları işleniyor...")

try:
    df_projeler = pd.read_csv("youth_portal_projeler_detayli.csv")
    df_projeler = df_projeler.dropna(subset=['tam_aciklama'])
except FileNotFoundError:
    raise

proje_beceri_listesi = []
start_time = time.time()

# İlerleme çubuğu gibi çalışması için sayaç
count = 0
total = len(df_projeler)

for text in df_projeler["tam_aciklama"].astype(str):
    try:
        ann = skill_extractor.annotate(text)
        skills = []
        for match in ann['results']['full_matches']:
            skills.append(match['doc_node_value'])
        for match in ann['results']['ngram_scored']:
            skills.append(match['doc_node_value'])
        
        proje_beceri_listesi.append(list(set([s.lower() for s in skills])))
    except:
        proje_beceri_listesi.append([])
    
    count += 1
    if count % 50 == 0:
        print(f"   ...{count}/{total} ilan işlendi.")

print(f"-> {total} ilan {time.time() - start_time:.2f} saniyede işlendi.")

df_projeler["cikarilan_beceriler"] = proje_beceri_listesi
df_projeler.to_pickle("IP1_Ciktisi_Projeler_Becerili.pkl")

print("\n--- İP 1 - Adım 4 TAMAMLANDI ---")
print(df_projeler[["link", "cikarilan_beceriler"]].head())

✅ 'skillNer' kütüphanesi başarıyla import edildi.
İP 1 - Adım 4 Başladı: Modeller yükleniyor...
loading full_matcher ...
loading abv_matcher ...
loading full_uni_matcher ...
loading low_form_matcher ...
loading token_matcher ...
✅ SkillExtractor başarıyla başlatıldı.

Adım 4.2: Aday CV metni işleniyor...
-> CV'den 13 adet beceri çıkarıldı.
Örnekler: ['social medium', 'c', 'html', 'sql server', 'c++']

Adım 4.3: Proje ilanları işleniyor...


/Users/acetiner/miniconda3/envs/tubitak-env/lib/python3.11/site-packages/skillNer/utils.py:99: UserWarning: [W008] Evaluating Token.similarity based on empty vectors.
  vec_similarity = token1.similarity(token2)


   ...50/952 ilan işlendi.
   ...100/952 ilan işlendi.
   ...150/952 ilan işlendi.
   ...200/952 ilan işlendi.
   ...250/952 ilan işlendi.
   ...300/952 ilan işlendi.
   ...350/952 ilan işlendi.
   ...400/952 ilan işlendi.
   ...450/952 ilan işlendi.
   ...500/952 ilan işlendi.
   ...550/952 ilan işlendi.
   ...600/952 ilan işlendi.
   ...650/952 ilan işlendi.
   ...700/952 ilan işlendi.
   ...750/952 ilan işlendi.
   ...800/952 ilan işlendi.
   ...850/952 ilan işlendi.
   ...900/952 ilan işlendi.
   ...950/952 ilan işlendi.
-> 952 ilan 1703.07 saniyede işlendi.

--- İP 1 - Adım 4 TAMAMLANDI ---
                                                link  \
0  https://youth.europa.eu/solidarity/opportunity...   
1  https://youth.europa.eu/solidarity/opportunity...   
2  https://youth.europa.eu/solidarity/opportunity...   
3  https://youth.europa.eu/solidarity/opportunity...   
4  https://youth.europa.eu/solidarity/opportunity...   

                                 cikarilan_beceriler  
0  [c

In [1]:
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 4.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 7.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]


In [2]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pandas as pd
import time
from tqdm import tqdm


model_name = "xlm-roberta-base"

print("İP 1 - Adım 5 Başladı: XLM-RoBERTa (ESCO Temeli) ile Vektörizasyon...")

# 1. Cihaz Ayarı
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Kullanılan cihaz: {device}")

# 2. Modeli Yükle (GÜNCELLENDİ)
model_name = "xlm-roberta-base" # Kesin çalışan, herkese açık model
print(f"Model yükleniyor: {model_name}...")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    print("✅ Model başarıyla yüklendi.")
except Exception as e:
    print(f"❌ Model yüklenirken hata oluştu: {e}")
    raise e

def get_esco_vectors(text_list):
    """Verilen beceri listesini vektörlere dönüştürür."""
    if not isinstance(text_list, list) or len(text_list) == 0:
        return np.zeros((1, 768)) # XLM-R-base boyutu 768'dir
        
    with torch.no_grad():
        inputs = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt").to(device)
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        return embeddings

# -------------------------------------------------------------------------
# ADIM 5.1: ADAY BECERİLERİNİ VEKTÖRİZE ET
# -------------------------------------------------------------------------
print("\nAdım 5.1: Aday becerileri vektörize ediliyor...")

try:
    df_cv_skills = pd.read_pickle("IP1_Ciktisi_Aday_Cikarilan_Beceriler.pkl")
    cv_skills_list = df_cv_skills["cikarilan_beceri_adi"].tolist()

    if cv_skills_list:
        aday_beceri_vektorleri = get_esco_vectors(cv_skills_list)
        print(f"-> {len(cv_skills_list)} aday becerisi vektörize edildi.")
        print(f"Vektör Matrisi Boyutu: {aday_beceri_vektorleri.shape}")
        np.save("IP1_Ciktisi_Aday_Vektorleri.npy", aday_beceri_vektorleri)
    else:
        print("UYARI: Aday beceri listesi boş!")
except FileNotFoundError:
    print("HATA: Aday beceri dosyası bulunamadı.")

# -------------------------------------------------------------------------
# ADIM 5.2: PROJE BECERİLERİNİ VEKTÖRİZE ET
# -------------------------------------------------------------------------
print("\nAdım 5.2: Proje ilanları vektörize ediliyor...")

try:
    df_projeler = pd.read_pickle("IP1_Ciktisi_Projeler_Becerili.pkl")
except FileNotFoundError:
    print("HATA: Proje dosyası bulunamadı.")
    raise

start_time = time.time()
df_projeler["beceri_vektorleri"] = df_projeler["cikarilan_beceriler"].apply(get_esco_vectors)
end_time = time.time()

print(f"-> {len(df_projeler)} proje {end_time - start_time:.2f} saniyede vektörize edildi.")

# Sonuçları göster (Link ile)
print(df_projeler[["link", "cikarilan_beceriler", "beceri_vektorleri"]].head())

# KAYDET
df_projeler.to_pickle("IP1_Ciktisi_Projeler_Vektorlu.pkl")

print("\n✅✅✅ İP 1 (VERİ TOPLAMA VE STANDARDİZASYON) TAMAMLANDI! ✅✅✅")

İP 1 - Adım 5 Başladı: XLM-RoBERTa (ESCO Temeli) ile Vektörizasyon...
Kullanılan cihaz: mps
Model yükleniyor: xlm-roberta-base...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

✅ Model başarıyla yüklendi.

Adım 5.1: Aday becerileri vektörize ediliyor...
-> 13 aday becerisi vektörize edildi.
Vektör Matrisi Boyutu: (13, 768)

Adım 5.2: Proje ilanları vektörize ediliyor...
-> 952 proje 67.58 saniyede vektörize edildi.
                                                link  \
0  https://youth.europa.eu/solidarity/opportunity...   
1  https://youth.europa.eu/solidarity/opportunity...   
2  https://youth.europa.eu/solidarity/opportunity...   
3  https://youth.europa.eu/solidarity/opportunity...   
4  https://youth.europa.eu/solidarity/opportunity...   

                                 cikarilan_beceriler  \
0  [com, teamwork, creative, hosting, food, creat...   
1  [psychological, teamwork, food, learning envir...   
2  [information system, cataloguing, geology envi...   
3  [research, food, data collection, reproductive...   
4  [english, digitalization, food, creativity, di...   

                                   beceri_vektorleri  
0  [[0.016297929, 0.0673945, 